# Chapitre 2.5 — Prétraitement des mammographies

Une mammographie brute ne peut pas être envoyée telle quelle à un modèle comme
GMIC : il attend des images de **taille fixe**, **sans le fond noir** autour du
sein, et dans une **plage d'intensité** précise. Quatre étapes transforment l'image
brute en ce format.

Ce notebook **s'exécute sur les images d'exemple livrées avec GMIC**
(`~/GMIC/sample_data/images/`, 16 mammographies) — pas besoin d'avoir téléchargé
RSNA. Si les données RSNA prétraitées sont montées dans `~/data`, le notebook les
utilise en priorité.

In [ ]:
import os, glob
import cv2, numpy as np
import matplotlib.pyplot as plt
from course_utils import flowchart

GMIC_H, GMIC_W = 2944, 1920   # taille d'entrée attendue par GMIC

# Décodeurs DICOM (utiles seulement si on lit du .dcm RSNA brut)
import importlib.util as u
for m in ('pylibjpeg_libjpeg', 'pylibjpeg_openjpeg'):
    print(m, 'OK' if u.find_spec(m) else 'MANQUANT')

flowchart([
    '1. DICOM -> PNG   (extrait les pixels, corrige MONOCHROME1)',
    '2. PKL GMIC       (liste examens + labels + chemins)',
    '3. Recadrage      (supprime le fond noir autour du sein)',
    '4. Resize 2944x1920 + uint8',
    'A l inference : flip des vues droites + normalisation mean/std',
], title='Pipeline de pretraitement')

In [ ]:
# Où trouver des images ? RSNA prétraité si dispo, sinon les exemples GMIC.
CANDIDATES = [
    os.path.expanduser('~/data/preprocess_image/cropped_images'),
    os.path.expanduser('~/data/rsna/train_images_png'),
    os.path.expanduser('~/GMIC/sample_data/images'),
]
IMG_DIR = next((d for d in CANDIDATES if os.path.isdir(d) and glob.glob(os.path.join(d, '**', '*.png'), recursive=True)), None)
assert IMG_DIR, 'Aucun dossier d images trouvé (GMIC devrait fournir sample_data).'
pngs = sorted(glob.glob(os.path.join(IMG_DIR, '**', '*.png'), recursive=True))
print('Dossier utilisé :', IMG_DIR)
print('Images trouvées  :', len(pngs))
print('Exemple          :', os.path.basename(pngs[0]))

## Étape 1 — DICOM → PNG

Les mammographies médicales sont stockées en **DICOM** (`.dcm`) : un format qui
encapsule bien plus que l'image (identité patient, paramètres scanner...). On en
extrait les pixels bruts.

Piège classique : certains scanners stockent les niveaux **inversés**
(`PhotometricInterpretation == 'MONOCHROME1'`) — sans correction, le tissu
mammaire apparaîtrait blanc sur fond noir, l'inverse de ce qu'attend GMIC.

In [ ]:
def dicom_to_array(path):
    import pydicom
    ds = pydicom.dcmread(path)
    arr = ds.pixel_array.astype(np.float32)
    if ds.PhotometricInterpretation == 'MONOCHROME1':
        arr = arr.max() - arr          # inverse les niveaux : blanc <-> noir
    return arr

dcms = glob.glob(os.path.expanduser('~/data/rsna/**/*.dcm'), recursive=True)[:1]
if dcms:
    arr = dicom_to_array(dcms[0])
    print('DICOM décodé :', dcms[0], '->', arr.shape, arr.dtype)
else:
    print('Aucun .dcm monté -> étape illustrée seulement (les exemples GMIC sont déjà en PNG).')

## Étape 2 — Le fichier PKL

GMIC n'accède pas aux fichiers image directement : il lit un **`.pkl`** qui décrit
chaque examen — chemins des 4 vues (L-CC, L-MLO, R-CC, R-MLO), labels
(`cancer_label['malignant']`), et métadonnées. C'est la colonne vertébrale que les
étapes suivantes enrichissent. Regardons celui livré avec GMIC.

In [ ]:
import pickle
pkl_path = os.path.expanduser('~/GMIC/sample_data/exam_list_before_cropping.pkl')
if os.path.isfile(pkl_path):
    with open(pkl_path, 'rb') as f:
        exams = pickle.load(f)
    print('Examens :', len(exams))
    e = exams[0]
    print('Vues :', {k: e[k] for k in ['L-CC', 'L-MLO', 'R-CC', 'R-MLO'] if k in e})
    print('Clés :', list(e.keys()))
else:
    print('pkl introuvable :', pkl_path)

## Étape 3 — Recadrage (supprimer le fond noir)

Une mammographie brute contient beaucoup de **fond noir**. GMIC, entraîné sur des
images recadrées, est moins performant si on le garde. En production, GMIC utilise
`crop_img_from_largest_connected` (`~/GMIC/src/cropping/crop_mammogram.py`) :
érosion → plus grande composante connexe → dilatation → bounding box.

Ci-dessous une version pédagogique courte (seuillage + bounding box) qui illustre
le principe et tourne sur n'importe quelle image.

In [ ]:
def crop_breast(img, thresh_frac=0.05):
    '''Retourne (image recadrée, (y0, y1, x0, x1)) en supprimant le fond noir.'''
    mask = img > (thresh_frac * float(img.max()))
    ys, xs = np.where(mask)
    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
    return img[y0:y1 + 1, x0:x1 + 1], (y0, y1, x0, x1)

raw = cv2.imread(pngs[0], cv2.IMREAD_UNCHANGED)   # garde la profondeur (souvent 16 bits)
print('Image brute :', raw.shape, raw.dtype, '| max =', int(raw.max()))
cropped, (y0, y1, x0, x1) = crop_breast(raw)
print('Recadrée    :', cropped.shape)

fig, ax = plt.subplots(1, 2, figsize=(11, 7))
ax[0].imshow(raw, cmap='gray')
ax[0].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, edgecolor='yellow', facecolor='none', lw=2))
ax[0].set_title(f'Brute {raw.shape[1]}x{raw.shape[0]} + zone recadrée'); ax[0].axis('off')
ax[1].imshow(cropped, cmap='gray')
ax[1].set_title(f'Recadrée {cropped.shape[1]}x{cropped.shape[0]}'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## Étape 4 — Resize 2944×1920 + uint8

Après recadrage les images ont des tailles variables (selon le scanner). GMIC ayant
été entraîné en **2944×1920**, on redimensionne tout à cette taille, puis on ramène
en **uint8** [0, 255].

**Deux interpolations** : `INTER_AREA` pour **réduire** (moyenne les pixels, évite
l'aliasing), `INTER_LINEAR` pour **agrandir** (bilinéaire, plus doux).

In [ ]:
h, w = cropped.shape[:2]
interp = cv2.INTER_AREA if (h > GMIC_H or w > GMIC_W) else cv2.INTER_LINEAR
resized = cv2.resize(cropped, (GMIC_W, GMIC_H), interpolation=interp)
f = resized.astype(np.float32); m = f.max()
u8 = (f / m * 255).astype(np.uint8) if m > 0 else f.astype(np.uint8)
print('Après resize :', u8.shape, u8.dtype, '| min/max =', int(u8.min()), int(u8.max()))
assert u8.shape == (GMIC_H, GMIC_W) and u8.dtype == np.uint8

plt.figure(figsize=(4, 6)); plt.imshow(u8, cmap='gray')
plt.title(f'Prête pour GMIC\n{GMIC_W}x{GMIC_H} uint8'); plt.axis('off'); plt.show()

## Récap

À l'issue des 4 étapes, chaque vue est un PNG **2944×1920 uint8**, fond noir retiré.
Restent deux opérations appliquées **à la volée à l'inférence** (inutile de les
stocker) :

1. **Flip horizontal** des vues droites (R-CC, R-MLO) → tous les seins « regardent »
   à gauche ;
2. **Normalisation mean/std** → distribution centrée, attendue par le réseau.

Chapitre suivant → `03_resnet18_breast_density.ipynb`.

## 🧪 Smoke test

Vérifie que le code clé de ce chapitre tourne **sans télécharger les données ni lancer le preprocessing complet** (entrées synthétiques).

In [ ]:
# 🧪 Smoke test — valide crop + resize + uint8 sur une image SYNTHÉTIQUE (aucun DICOM/PNG réel).
import numpy as np, cv2

GMIC_H, GMIC_W = 2944, 1920
if 'crop_breast' not in dir():                # permet d'exécuter ce test seul
    def crop_breast(img, thresh_frac=0.05):
        mask = img > (thresh_frac * float(img.max()))
        ys, xs = np.where(mask)
        y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
        return img[y0:y1 + 1, x0:x1 + 1], (y0, y1, x0, x1)

# "Sein" = disque brillant sur un grand fond noir
_fake = np.zeros((1000, 700), np.uint16)
_yy, _xx = np.ogrid[:1000, :700]
_fake[(_yy - 500) ** 2 + (_xx - 350) ** 2 < 200 ** 2] = 4000

_cropped, _ = crop_breast(_fake)
assert _cropped.shape[0] < _fake.shape[0] and _cropped.shape[1] < _fake.shape[1], 'le crop ne réduit rien'
assert _cropped.max() == _fake.max(), 'le crop a perdu le signal'

_h, _w = _cropped.shape[:2]
_interp = cv2.INTER_AREA if (_h > GMIC_H or _w > GMIC_W) else cv2.INTER_LINEAR
_resized = cv2.resize(_cropped, (GMIC_W, GMIC_H), interpolation=_interp)
_f = _resized.astype(np.float32); _m = _f.max()
_u8 = (_f / _m * 255).astype(np.uint8) if _m > 0 else _f.astype(np.uint8)
assert _u8.shape == (GMIC_H, GMIC_W) and _u8.dtype == np.uint8, (_u8.shape, _u8.dtype)
print('✅ crop_breast + resize 2944x1920 + uint8 OK (image synthétique, sans preprocessing réel).')